# NB-S: Catalytic Organic Synthesis --- Supplementary Capstone

> **Chapter Map:** This notebook is a **supplementary capstone exercise** aligned with the Appendix on Catalytic Organic Synthesis. It integrates concepts from all six chapters.

**Prerequisites:** This capstone integrates concepts from all six chapters: Arrhenius kinetics (Ch 2), TOF/TON (Ch 3), transport effects (Ch 4), selectivity (Ch 5), and catalyst materials (Ch 6).

## Learning Objectives

After completing this notebook, you will be able to:

1. Calculate turnover number (TON) and turnover frequency (TOF) from experimental conversion-time data and understand how these metrics depend on catalyst loading
2. Predict enantiomeric excess (ee) from transition state energy differences using the Boltzmann distribution and explore the temperature-selectivity tradeoff
3. Model first-order catalyst deactivation kinetics, extract deactivation parameters from noisy data using curve fitting, and predict total catalyst productivity
4. Evaluate green chemistry metrics (atom economy, E-factor) for competing synthetic routes
5. Combine multiple catalyst performance metrics into a comprehensive evaluation framework

## Background

This capstone notebook integrates concepts from the entire course---adsorption, kinetics, temperature dependence, reactor design, transport, and selectivity---in the context of catalytic organic synthesis. We focus on Pd-catalyzed cross-coupling reactions (Suzuki, Heck, Sonogashira), asymmetric catalysis, and green chemistry metrics.

### Key Equations

**Turnover Number and Frequency:**
$$\text{TON} = \frac{n_{\text{product}}}{n_{\text{catalyst}}} \qquad \text{TOF} = \frac{\text{TON}}{t}$$

**Enantiomeric Excess from Transition State Energies:**
$$\frac{[R]}{[S]} = \exp\left(\frac{\Delta\Delta G^{\ddagger}}{RT}\right) \qquad \text{ee} = \frac{\text{ratio} - 1}{\text{ratio} + 1} \times 100\%$$

**First-Order Catalyst Deactivation:**
$$\text{TOF}(t) = \text{TOF}_0 \cdot e^{-k_d t} \qquad \text{TON}_{\text{final}} = \frac{\text{TOF}_0}{k_d}$$

**Green Chemistry Metrics:**
$$\text{Atom Economy} = \frac{M_{\text{product}}}{\sum M_{\text{reactants}}} \times 100\% \qquad \text{E-factor} = \frac{\text{kg waste}}{\text{kg product}}$$

---
## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from scipy.optimize import curve_fit

# Set default plot style for publication-quality figures
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Physical constant
R = 8.314  # Gas constant in J/(mol*K)

# Colorblind-safe color palette (Wong, 2011)
COLORS = {
    'blue': '#0072B2',
    'orange': '#E69F00',
    'green': '#009E73',
    'vermillion': '#D55E00',
    'skyblue': '#56B4E9',
    'purple': '#CC79A7'
}

print("Setup complete. Libraries imported successfully.")

---
## Part 1: TON and TOF Calculations from Suzuki Coupling Data

In Pd-catalyzed Suzuki coupling, an aryl halide reacts with a boronic acid to form a biaryl product:

$$\text{Ar--Br} + \text{Ar'--B(OH)}_2 \xrightarrow[\text{base}]{\text{Pd catalyst}} \text{Ar--Ar'} + \text{B(OH)}_3 + \text{NaBr}$$

The **turnover number (TON)** measures catalyst durability---how many product molecules each catalyst molecule produces:

$$\text{TON} = \frac{n_{\text{product}}}{n_{\text{catalyst}}} = \frac{X}{\text{cat\_loading}}$$

where $X$ is the fractional conversion and cat\_loading is the catalyst mole fraction.

The **turnover frequency (TOF)** measures catalyst activity---how fast each catalytic cycle turns over:

$$\text{TOF} = \frac{d(\text{TON})}{dt}$$

We will analyze conversion-time data for three different Pd loadings (0.1, 1.0, and 5.0 mol%) and explore how these metrics distinguish intrinsic activity from loading effects.

### 1.1 Loading and Inspecting the Experimental Data

In [ ]:
# Load Suzuki coupling conversion-time data
df_suzuki = pd.read_csv('data/suzuki_coupling_data.csv', comment='#')

print("Suzuki Coupling Data: Conversion vs Time at Three Pd Loadings")
print("=" * 65)
print(f"Shape: {df_suzuki.shape[0]} time points x {df_suzuki.shape[1]} columns")
print(f"Columns: {list(df_suzuki.columns)}")
print()
print(df_suzuki.to_string(index=False))

### 1.2 Implementing the TON/TOF Calculation

In [ ]:
def calculate_ton_tof(time, conversion, cat_loading):
    """
    Calculate turnover number and turnover frequency from conversion-time data.
    
    Parameters
    ----------
    time : array-like
        Time points (h)
    conversion : array-like
        Fractional conversion at each time point (0 to 1)
    cat_loading : float
        Catalyst loading as mole fraction (e.g., 0.01 for 1 mol%)
    
    Returns
    -------
    ton : ndarray
        Turnover number at each time point (dimensionless)
    tof : ndarray
        Turnover frequency at each time point (h^-1)
    
    Notes
    -----
    TON = conversion / cat_loading
    TOF = d(TON)/dt, computed via centered finite differences (np.gradient)
    
    At complete conversion (X=1), TON = 1/cat_loading.
    For example, 1 mol% loading gives maximum TON = 100.
    """
    time = np.asarray(time)
    conversion = np.asarray(conversion)
    ton = conversion / cat_loading
    tof = np.gradient(ton, time)
    return ton, tof


# Test with a simple case: at 50% conversion with 1 mol% loading
# TON should be 0.5 / 0.01 = 50
test_ton, _ = calculate_ton_tof([0, 1], [0, 0.5], 0.01)
print(f"Test: 50% conversion at 1 mol% loading -> TON = {test_ton[-1]:.0f}")
print(f"Expected: TON = 50")

### 1.3 Computing TON and TOF for Each Loading

In [ ]:
# Catalyst loadings as mole fractions
loadings = {'0.1 mol%': 0.001, '1.0 mol%': 0.01, '5.0 mol%': 0.05}
conv_columns = {
    '0.1 mol%': 'conversion_0p1',
    '1.0 mol%': 'conversion_1p0',
    '5.0 mol%': 'conversion_5p0'
}

time = df_suzuki['time_h'].values

# Store results for plotting
results = {}
for label, loading in loadings.items():
    conv = df_suzuki[conv_columns[label]].values
    ton, tof = calculate_ton_tof(time, conv, loading)
    results[label] = {'conversion': conv, 'ton': ton, 'tof': tof}

# Print summary at final time point
print("Summary at t = {:.0f} h".format(time[-1]))
print("=" * 55)
print(f"{'Loading':>12} | {'Conversion':>12} | {'TON':>10} | {'TOF (h-1)':>12}")
print("-" * 55)
for label in loadings:
    r = results[label]
    print(f"{label:>12} | {r['conversion'][-1]:>12.3f} | "
          f"{r['ton'][-1]:>10.0f} | {r['tof'][-1]:>12.1f}")

### 1.4 Visualizing Conversion, TON, and TOF

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

colors_list = [COLORS['blue'], COLORS['orange'], COLORS['green']]
markers = ['o', 's', '^']
linestyles = ['-', '--', '-.']

for idx, (label, r) in enumerate(results.items()):
    color = colors_list[idx]
    marker = markers[idx]
    ls = linestyles[idx]
    
    # Panel 1: Conversion vs Time
    axes[0].plot(time, r['conversion'], marker=marker, color=color,
                 linestyle=ls, markersize=6, markeredgecolor='black',
                 markeredgewidth=0.8, label=label)
    
    # Panel 2: TON vs Time
    axes[1].plot(time, r['ton'], marker=marker, color=color,
                 linestyle=ls, markersize=6, markeredgecolor='black',
                 markeredgewidth=0.8, label=label)
    
    # Panel 3: TOF vs Time (skip t=0 to avoid edge artifacts)
    mask = time > 0
    axes[2].plot(time[mask], r['tof'][mask], marker=marker, color=color,
                 linestyle=ls, markersize=6, markeredgecolor='black',
                 markeredgewidth=0.8, label=label)

# Panel 1 formatting
axes[0].set_xlabel('Time (h)')
axes[0].set_ylabel('Conversion, $X$ (dimensionless)')
axes[0].set_title('(a) Conversion vs Time')
axes[0].set_ylim(-0.02, 1.05)
axes[0].legend(loc='lower right')

# Panel 2 formatting
axes[1].set_xlabel('Time (h)')
axes[1].set_ylabel('Turnover Number, TON')
axes[1].set_title('(b) TON vs Time')
axes[1].legend(loc='center right')

# Panel 3 formatting
axes[2].set_xlabel('Time (h)')
axes[2].set_ylabel('Turnover Frequency, TOF (h$^{-1}$)')
axes[2].set_title('(c) TOF vs Time')
axes[2].legend(loc='upper right')

plt.tight_layout()
plt.show()

### Key Observations

1. **Conversion (panel a)**: Higher catalyst loading gives faster conversion, as expected. At 5 mol%, the reaction is essentially complete within 4 h.

2. **TON (panel b)**: Despite reaching higher conversion faster, the 5.0 mol% loading achieves the *lowest* TON because each Pd atom does less work. The 0.1 mol% loading reaches much higher TON---each catalyst molecule converts far more substrate molecules.

3. **TOF (panel c)**: The initial TOF values are similar across loadings, indicating that the *intrinsic activity* per Pd center is similar. TOF is an intensive property of the catalyst, while TON depends on the loading.

This is a fundamental insight: **TOF reflects intrinsic catalyst quality, while TON reflects both catalyst quality and loading.**

---
## Part 2: Enantiomeric Excess from Transition State Energy Differences

In asymmetric catalysis, a chiral catalyst creates two diastereomeric transition states of different energy. The product enantiomer ratio is determined by the Boltzmann distribution:

$$\frac{[R]}{[S]} = \exp\left(\frac{\Delta\Delta G^{\ddagger}}{RT}\right)$$

where $\Delta\Delta G^{\ddagger} = \Delta G^{\ddagger}_S - \Delta G^{\ddagger}_R > 0$ when the $R$-enantiomer is favored.

The enantiomeric excess is then:

$$\text{ee} = \frac{\text{ratio} - 1}{\text{ratio} + 1} \times 100\%$$

Small energy differences translate to large selectivity at low temperature because the Boltzmann factor becomes more discriminating when $RT$ is small.

### 2.1 Implementing the ee Calculation

In [ ]:
def ee_from_ddG(ddG, T):
    """
    Calculate enantiomeric excess from the energy difference between
    diastereomeric transition states.
    
    Parameters
    ----------
    ddG : float or array-like
        Energy difference between diastereomeric transition states (J/mol).
        Positive when the R-enantiomer is favored.
    T : float or array-like
        Temperature (K)
    
    Returns
    -------
    float or ndarray
        Enantiomeric excess as a percentage (0 to 100)
    
    Notes
    -----
    Uses the Boltzmann distribution:
        ratio = exp(ddG / (R*T))
        ee = (ratio - 1) / (ratio + 1) * 100
    
    At ddG = 0: ee = 0% (racemic)
    At ddG >> RT: ee -> 100% (enantiopure)
    At high T: ee -> 0% (thermal scrambling)
    """
    ddG = np.asarray(ddG, dtype=float)
    T = np.asarray(T, dtype=float)
    ratio = np.exp(ddG / (R * T))
    ee = (ratio - 1) / (ratio + 1) * 100
    return ee


# Verify against worked example from lecture notes:
# ddG = 5.0 kJ/mol at 25 degC should give ee ~ 76%
ee_test = ee_from_ddG(5000, 298.15)
print(f"Test: ddG = 5.0 kJ/mol at 25 degC")
print(f"  Calculated ee = {ee_test:.1f}%")
print(f"  Expected ee ~ 76% (from lecture notes)")

### 2.2 Reproducing the Lecture Notes Table

The lecture notes provide a reference table showing ee at 25 $^\circ$C and $-78$ $^\circ$C for various $\Delta\Delta G^{\ddagger}$ values. Let us reproduce and extend it.

In [ ]:
# =============================================
# ADJUSTABLE PARAMETERS
# =============================================
# Change ddG values or temperatures to explore
# how the ee landscape shifts.
ddG_values_kJ = [2.0, 4.0, 6.0, 8.0]  # kJ/mol
T_25C = 298.15      # K
T_minus78C = 195.15  # K (-78 degC, dry ice bath)
T_minus40C = 233.15  # K (-40 degC)
# =============================================

# Reproduce the table from the lecture notes
print("Enantiomeric Excess vs Transition State Energy Difference")
print("=" * 70)
print(f"{'ddG (kJ/mol)':>14} | {'ee at 25 C':>12} | "
      f"{'ee at -40 C':>13} | {'ee at -78 C':>13}")
print("-" * 70)

for ddG_kJ in ddG_values_kJ:
    ddG_J = ddG_kJ * 1000  # Convert kJ/mol to J/mol
    ee_25 = ee_from_ddG(ddG_J, T_25C)
    ee_m40 = ee_from_ddG(ddG_J, T_minus40C)
    ee_m78 = ee_from_ddG(ddG_J, T_minus78C)
    print(f"{ddG_kJ:>14.1f} | {ee_25:>11.0f}% | "
          f"{ee_m40:>12.0f}% | {ee_m78:>12.0f}%")

print("-" * 70)
print("\nKey insight: Lowering temperature from 25 C to -78 C substantially")
print("improves ee, especially for modest energy differences (2-6 kJ/mol).")

### 2.3 Enantiomeric Excess Heatmap

Let us create a heatmap showing ee as a function of both $\Delta\Delta G^{\ddagger}$ and temperature. This provides a complete picture of the selectivity-temperature landscape.

**Concept Check:** For $\Delta\Delta G^{\ddagger} = 6$ kJ/mol, predict: at what temperature does ee drop below 90%? Estimate using $RT \approx 2.5$ kJ/mol at 300 K, then check the heatmap.

In [ ]:
# Create ee heatmap
ddG_range = np.linspace(2000, 12000, 200)  # 2 to 12 kJ/mol in J/mol
T_range = np.linspace(195, 373, 200)  # -78 C to 100 C in K

ddG_grid, T_grid = np.meshgrid(ddG_range, T_range)
ee_grid = ee_from_ddG(ddG_grid, T_grid)

fig, ax = plt.subplots(figsize=(10, 7))

# Heatmap (viridis is colorblind-safe)
c = ax.pcolormesh(ddG_range / 1000, T_range - 273.15, ee_grid,
                  cmap='viridis', shading='auto', vmin=0, vmax=100)

# Colorbar
cbar = fig.colorbar(c, ax=ax, label='Enantiomeric Excess, ee (%)')

# Contour lines at important thresholds
contour_levels = [50, 80, 90, 95, 99]
CS = ax.contour(ddG_range / 1000, T_range - 273.15, ee_grid,
                levels=contour_levels, colors='white',
                linewidths=1.5, linestyles='--')
ax.clabel(CS, inline=True, fontsize=10, fmt='%d%%')

# Mark the lecture notes data points
for ddG_kJ in [2.0, 4.0, 6.0, 8.0]:
    for T_C in [25, -78]:
        ax.plot(ddG_kJ, T_C, 'wo', markersize=6, markeredgecolor='black')

# Mark pharmaceutical threshold (ee > 99%)
ax.annotate('Pharmaceutical\nthreshold\n(ee > 99%)',
            xy=(9, -40), fontsize=10, ha='center',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Labels
ax.set_xlabel('$\\Delta\\Delta G^{\\ddagger}$ (kJ/mol)')
ax.set_ylabel('Temperature ($^\\circ$C)')
ax.set_title('Enantiomeric Excess: Selectivity-Temperature Landscape')

plt.tight_layout()
plt.show()

print("Interpretation:")
print("- Bright (yellow) region: high ee (good enantioselectivity)")
print("- Dark (purple) region: low ee (poor selectivity, near racemic)")
print("- Moving right (larger ddG) or down (lower T) improves ee")

### 2.4 Finding the Critical Temperature for a Target ee

For pharmaceutical applications, ee > 99% is often required. Given a catalyst with known $\Delta\Delta G^{\ddagger}$, what temperature is needed?

In [ ]:
# Analytical inversion: solve ee = target for T
# ratio = (100 + ee) / (100 - ee)
# T = ddG / (R * ln(ratio))

def critical_temperature(ddG, ee_target=99.0):
    """
    Find the maximum temperature that achieves a target ee.
    
    Parameters
    ----------
    ddG : float
        Energy difference (J/mol)
    ee_target : float
        Target enantiomeric excess (%)
    
    Returns
    -------
    float
        Maximum temperature (K)
    """
    ratio = (100 + ee_target) / (100 - ee_target)
    return ddG / (R * np.log(ratio))


print("Maximum Temperature for ee > 99%")
print("=" * 45)
print(f"{'ddG (kJ/mol)':>14} | {'T_max (K)':>10} | {'T_max (C)':>10}")
print("-" * 45)

for ddG_kJ in [2, 4, 6, 8, 10, 12]:
    T_max = critical_temperature(ddG_kJ * 1000, 99.0)
    print(f"{ddG_kJ:>14} | {T_max:>10.1f} | {T_max - 273.15:>10.1f}")

print("\nKey insight: For ddG < 6 kJ/mol, cryogenic temperatures")
print("are required to achieve pharmaceutical-grade ee.")

---
## Part 3: Catalyst Deactivation Kinetics

Catalysts lose activity over time due to poisoning, aggregation, or ligand degradation. For first-order deactivation, the TOF decays exponentially:

$$\text{TOF}(t) = \text{TOF}_0 \cdot e^{-k_d t}$$

The total number of turnovers over the catalyst lifetime is:

$$\text{TON}_{\text{final}} = \int_0^{\infty} \text{TOF}(t)\, dt = \frac{\text{TOF}_0}{k_d}$$

This shows that TON depends on the *ratio* of initial activity to deactivation rate. A moderately active but stable catalyst may outperform a highly active but unstable one.

### 3.1 The Deactivation Model

In [ ]:
def deactivation_model(time, TOF0, kd):
    """
    First-order catalyst deactivation model.
    
    Parameters
    ----------
    time : float or array-like
        Time points (h)
    TOF0 : float
        Initial turnover frequency (h^-1)
    kd : float
        Deactivation rate constant (h^-1)
    
    Returns
    -------
    ndarray
        TOF at each time point (h^-1)
    
    Notes
    -----
    TOF(t) = TOF0 * exp(-kd * t)
    
    The cumulative TON at time t is:
        TON(t) = (TOF0 / kd) * (1 - exp(-kd * t))
    
    The total TON as t -> infinity is:
        TON_final = TOF0 / kd
    
    Half-life: t_1/2 = ln(2) / kd
    """
    time = np.asarray(time, dtype=float)
    return TOF0 * np.exp(-kd * time)


# Quick demonstration
t_demo = np.linspace(0, 30, 100)
tof_demo = deactivation_model(t_demo, TOF0=500, kd=0.15)
print(f"TOF at t=0: {tof_demo[0]:.0f} h^-1")
print(f"TOF at t=10 h: {deactivation_model(10, 500, 0.15):.0f} h^-1")
print(f"Half-life: {np.log(2)/0.15:.1f} h")
print(f"Predicted total TON: {500/0.15:.0f}")

### 3.2 Fitting Experimental Deactivation Data

Real experimental data contains noise. We load TOF vs time measurements and fit the deactivation model using `scipy.optimize.curve_fit`.

In [ ]:
# Load experimental TOF decay data
df_deact = pd.read_csv('data/tof_deactivation_data.csv', comment='#')

t_data = df_deact['time_h'].values
tof_data = df_deact['tof_per_hour'].values

print("Deactivation Data:")
print(df_deact.to_string(index=False))

In [ ]:
# Fit the deactivation model to the noisy data
popt, pcov = curve_fit(
    deactivation_model, t_data, tof_data,
    p0=[500, 0.1],  # Initial guesses for TOF0 and kd
    bounds=([0, 0], [np.inf, np.inf])  # Both parameters must be positive
)

TOF0_fit, kd_fit = popt
TOF0_err, kd_err = np.sqrt(np.diag(pcov))  # Standard errors

# Derived quantities
TON_final = TOF0_fit / kd_fit
half_life = np.log(2) / kd_fit

# Error propagation for TON = TOF0/kd
# d(TON)/d(TOF0) = 1/kd, d(TON)/d(kd) = -TOF0/kd^2
TON_err = TON_final * np.sqrt(
    (TOF0_err / TOF0_fit)**2 + (kd_err / kd_fit)**2
)

print("Deactivation Model Fit Results")
print("=" * 50)
print(f"TOF_0 = {TOF0_fit:.1f} +/- {TOF0_err:.1f} h^-1")
print(f"k_d   = {kd_fit:.4f} +/- {kd_err:.4f} h^-1")
print(f"")
print(f"Derived Quantities:")
print(f"  Total TON = TOF_0/k_d = {TON_final:.0f} +/- {TON_err:.0f}")
print(f"  Half-life = ln(2)/k_d = {half_life:.1f} h")

### 3.3 Visualization: Data, Fit, and Cumulative TON

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# --- Left panel: TOF vs Time with fit ---
t_fit = np.linspace(0, 25, 200)
tof_fit = deactivation_model(t_fit, TOF0_fit, kd_fit)

# Plot fit first (behind data)
ax1.plot(t_fit, tof_fit, '-', color=COLORS['vermillion'], linewidth=2.5,
         label=f'Fit: TOF$_0$ = {TOF0_fit:.0f} h$^{{-1}}$, '
               f'$k_d$ = {kd_fit:.3f} h$^{{-1}}$')

# Plot data points
ax1.plot(t_data, tof_data, 'o', color=COLORS['blue'], markersize=8,
         markeredgecolor='black', markeredgewidth=1,
         label='Experimental data')

# Mark half-life
ax1.axvline(x=half_life, color='gray', linestyle=':', linewidth=1.5, alpha=0.7)
ax1.annotate(f'$t_{{1/2}}$ = {half_life:.1f} h',
             xy=(half_life, TOF0_fit * 0.5),
             xytext=(half_life + 2, TOF0_fit * 0.55),
             fontsize=11, color='gray',
             arrowprops=dict(arrowstyle='->', color='gray'))

ax1.set_xlabel('Time (h)')
ax1.set_ylabel('Turnover Frequency, TOF (h$^{-1}$)')
ax1.set_title('(a) Catalyst Deactivation: TOF Decay')
ax1.legend(loc='upper right', fontsize=10)
ax1.set_ylim(0, None)

# --- Right panel: Cumulative TON ---
# Analytical cumulative TON
ton_cumulative = (TOF0_fit / kd_fit) * (1 - np.exp(-kd_fit * t_fit))

ax2.plot(t_fit, ton_cumulative, '-', color=COLORS['green'], linewidth=2.5,
         label='Cumulative TON')

# Mark the asymptotic TON
ax2.axhline(y=TON_final, color='gray', linestyle='--', linewidth=1.5, alpha=0.7)
ax2.annotate(f'TON$_{{\\mathrm{{final}}}}$ = {TON_final:.0f}',
             xy=(20, TON_final), xytext=(15, TON_final * 0.85),
             fontsize=12, color='gray',
             arrowprops=dict(arrowstyle='->', color='gray'))

# Shade the area to represent total TON
ax_fill_t = np.linspace(0, 25, 200)
ax_fill_tof = deactivation_model(ax_fill_t, TOF0_fit, kd_fit)

ax2.set_xlabel('Time (h)')
ax2.set_ylabel('Cumulative Turnover Number, TON')
ax2.set_title('(b) Cumulative Catalyst Productivity')
ax2.legend(loc='lower right', fontsize=11)
ax2.set_ylim(0, None)

plt.tight_layout()
plt.show()

# Verify: numerical integration vs analytical
ton_numerical = np.trapezoid(tof_fit, t_fit)
print(f"\nVerification: Numerical integral of TOF = {ton_numerical:.0f}")
print(f"Analytical TON_final = TOF0/kd = {TON_final:.0f}")
print(f"(Small difference because numerical integral is over finite time)")

### Interpretation

1. **TOF decay**: The exponential decay fit matches the data well, confirming first-order deactivation kinetics. The catalyst loses half its activity every ~4.6 h.

2. **Cumulative TON**: The total productivity asymptotes to TOF$_0$/k$_d$. After ~5 half-lives (~23 h), the catalyst has delivered >97% of its total turnovers.

3. **Activity vs stability tradeoff**: The relationship TON = TOF$_0$/k$_d$ shows that doubling the initial activity (TOF$_0$) is equivalent to halving the deactivation rate (k$_d$) in terms of total productivity. For industrial applications, improving stability is often more impactful.

---
## Part 4: Green Chemistry Metrics

Sustainable synthesis requires quantitative metrics to compare routes. Two key metrics are:

**Atom Economy** measures what fraction of reactant atoms end up in the desired product:

$$\text{Atom Economy} = \frac{M_{\text{product}}}{\sum M_{\text{reactants}}} \times 100\%$$

**E-factor** measures total waste per unit product:

$$\text{E-factor} = \frac{\text{mass of waste}}{\text{mass of product}}$$

### 4.1 Implementing Green Chemistry Functions

In [ ]:
def atom_economy(MW_product, sum_MW_reactants):
    """
    Calculate atom economy for a reaction.
    
    Parameters
    ----------
    MW_product : float
        Molecular weight of the desired product (g/mol)
    sum_MW_reactants : float
        Sum of molecular weights of all reactants (g/mol)
    
    Returns
    -------
    float
        Atom economy as a percentage (0 to 100)
    
    Notes
    -----
    Atom economy measures theoretical efficiency of a reaction.
    Does not account for solvents, excess reagents, or workup waste.
    100% atom economy means all reactant atoms appear in the product.
    """
    return (MW_product / sum_MW_reactants) * 100


def e_factor(mass_waste, mass_product):
    """
    Calculate the environmental factor (E-factor).
    
    Parameters
    ----------
    mass_waste : float
        Total mass of waste produced (kg)
    mass_product : float
        Mass of desired product (kg)
    
    Returns
    -------
    float
        E-factor (dimensionless, lower is better)
    
    Notes
    -----
    Typical E-factors by sector:
    - Oil refining: < 0.1
    - Bulk chemicals: 1-5
    - Fine chemicals: 5-50
    - Pharmaceuticals: 25-100+
    """
    return mass_waste / mass_product


# Test: Suzuki coupling atom economy
# C6H5Br (157) + C6H5B(OH)2 (122) -> C6H5-C6H5 (154) + B(OH)3 + NaBr
ae_suzuki = atom_economy(154, 157 + 122)
print(f"Suzuki coupling atom economy: {ae_suzuki:.1f}%")

### 4.2 Comparing Suzuki Coupling vs Grignard Route

We compare two routes to biphenyl (C$_6$H$_5$--C$_6$H$_5$, MW = 154 g/mol):

**Route A: Suzuki Coupling (catalytic)**
$$\text{C}_6\text{H}_5\text{Br} + \text{C}_6\text{H}_5\text{B(OH)}_2 \xrightarrow{\text{Pd cat, base}} \text{C}_{12}\text{H}_{10} + \text{B(OH)}_3 + \text{NaBr}$$

**Route B: Grignard/Kumada Route (stoichiometric metal)**
$$\text{C}_6\text{H}_5\text{Br} + \text{Mg} \to \text{C}_6\text{H}_5\text{MgBr}$$
$$\text{C}_6\text{H}_5\text{MgBr} + \text{C}_6\text{H}_5\text{Br} \to \text{C}_{12}\text{H}_{10} + \text{MgBr}_2$$

Net: $2 \text{C}_6\text{H}_5\text{Br} + \text{Mg} \to \text{C}_{12}\text{H}_{10} + \text{MgBr}_2$

In [ ]:
# =============================================
# ADJUSTABLE PARAMETERS
# =============================================
# Molecular weights (g/mol). Adjust if comparing
# different substrates or coupling partners.
MW_biphenyl = 154.21
MW_PhBr = 157.01
MW_PhBOH2 = 121.93
MW_Mg = 24.31
MW_MgBr2 = 184.11   # Byproduct from Grignard

# Practical E-factors (including solvents, excess reagents, workup)
ef_suzuki = 5.0    # Suzuki: catalytic, mild conditions
ef_grignard = 15.0  # Grignard: anhydrous conditions, quenching
# =============================================

# Route A: Suzuki Coupling
# Note: Stoichiometric base (e.g., NaOH, MW=40.00) is also consumed but
# excluded here for simplicity.  Including it would lower AE to ~48%.
reactants_suzuki = MW_PhBr + MW_PhBOH2
ae_suzuki = atom_economy(MW_biphenyl, reactants_suzuki)

# Route B: Grignard
reactants_grignard = 2 * MW_PhBr + MW_Mg
ae_grignard = atom_economy(MW_biphenyl, reactants_grignard)

print("Green Chemistry Comparison: Routes to Biphenyl")
print("=" * 55)
print(f"{'Metric':>25} | {'Suzuki':>12} | {'Grignard':>12}")
print("-" * 55)
print(f"{'Atom Economy (%)':>25} | {ae_suzuki:>12.1f} | {ae_grignard:>12.1f}")
print(f"{'E-factor (practical)':>25} | {ef_suzuki:>12.1f} | {ef_grignard:>12.1f}")
print(f"{'Catalyst loading':>25} | {'0.5-5 mol%':>12} | {'1 equiv Mg':>12}")
print(f"{'Conditions':>25} | {'Air/H2O ok':>12} | {'Anhydrous':>12}")
print()
print("Note: Suzuki AE excludes stoichiometric base (NaOH).")
print("Including base: AE ~ 48%. Qualitative ranking unchanged.")

In [ ]:
# Bar chart comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

routes = ['Suzuki\nCoupling', 'Grignard\nRoute']
x = np.arange(len(routes))
bar_width = 0.5

# Panel 1: Atom Economy
bars1 = ax1.bar(x, [ae_suzuki, ae_grignard], bar_width,
               color=[COLORS['blue'], COLORS['orange']],
               edgecolor='black', linewidth=1.5)

# Add value labels on bars
for bar, val in zip(bars1, [ae_suzuki, ae_grignard]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=13,
             fontweight='bold')

ax1.set_ylabel('Atom Economy (%)')
ax1.set_title('(a) Atom Economy Comparison')
ax1.set_xticks(x)
ax1.set_xticklabels(routes)
ax1.set_ylim(0, 70)
ax1.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
ax1.text(1.3, 51, '50% threshold', fontsize=10, color='gray')

# Panel 2: E-factor
bars2 = ax2.bar(x, [ef_suzuki, ef_grignard], bar_width,
               color=[COLORS['blue'], COLORS['orange']],
               edgecolor='black', linewidth=1.5)

for bar, val in zip(bars2, [ef_suzuki, ef_grignard]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.0f}', ha='center', va='bottom', fontsize=13,
             fontweight='bold')

ax2.set_ylabel('E-factor (kg waste / kg product)')
ax2.set_title('(b) E-factor Comparison (lower is better)')
ax2.set_xticks(x)
ax2.set_xticklabels(routes)
ax2.set_ylim(0, 20)

# Add typical range annotations
ax2.axhspan(1, 5, alpha=0.1, color=COLORS['green'])
ax2.text(1.6, 3, 'Fine chemicals\ntypical range', fontsize=9,
         color=COLORS['green'], ha='center')

plt.tight_layout()
plt.show()

print("The Suzuki coupling wins on both metrics:")
print(f"  Atom economy: {ae_suzuki:.1f}% vs {ae_grignard:.1f}%")
print(f"  E-factor: {ef_suzuki:.0f} vs {ef_grignard:.0f}")

---
## Part 5: Comprehensive Catalyst Evaluation

In practice, catalyst selection requires balancing multiple performance metrics simultaneously. We define three hypothetical catalyst systems and compare them using a radar (spider) chart.

| Metric | Cat A: Pd(PPh$_3$)$_4$ | Cat B: Pd/SPhos | Cat C: Pd-BINAP |
|--------|------------------------|------------------|-----------------|
| TON | 20 | 5000 | 1000 |
| TOF (h$^{-1}$) | 10 | 500 | 200 |
| ee (%) | 0 (achiral) | 0 (achiral) | 98 |
| E-factor | 15 | 5 | 8 |
| Cost (\$/mmol) | 5 | 2 | 20 |

In [ ]:
# Define catalyst performance data
catalysts = {
    'Pd(PPh$_3$)$_4$': {
        'TON': 20, 'TOF': 10, 'ee': 0,
        'E-factor': 15, 'Cost': 5
    },
    'Pd/SPhos': {
        'TON': 5000, 'TOF': 500, 'ee': 0,
        'E-factor': 5, 'Cost': 2
    },
    'Pd-BINAP': {
        'TON': 1000, 'TOF': 200, 'ee': 98,
        'E-factor': 8, 'Cost': 20
    }
}

# Normalize metrics to 0-1 scale for radar chart
# Higher is always better after normalization
metrics = ['TON', 'TOF', 'ee', 'E-factor', 'Cost']
max_vals = {'TON': 5000, 'TOF': 500, 'ee': 100,
            'E-factor': 20, 'Cost': 25}

# For E-factor and Cost, lower is better, so we invert
invert = {'E-factor', 'Cost'}

def normalize_metrics(cat_data, max_vals, invert_set):
    """Normalize metrics to [0, 1] scale where 1 is always best."""
    normalized = []
    for m in metrics:
        val = cat_data[m] / max_vals[m]
        if m in invert_set:
            val = 1 - val  # Invert so lower original = higher score
        normalized.append(min(val, 1.0))  # Cap at 1
    return normalized

# Create radar chart
labels = ['TON\n(durability)', 'TOF\n(activity)', 'ee\n(selectivity)',
          'E-factor\n(lower=better)', 'Cost\n(lower=better)']
N = len(labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # Close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

colors_radar = [COLORS['blue'], COLORS['orange'], COLORS['green']]
linestyles_radar = ['-', '--', '-.']

for idx, (cat_name, cat_data) in enumerate(catalysts.items()):
    values = normalize_metrics(cat_data, max_vals, invert)
    values += values[:1]  # Close the polygon
    
    ax.plot(angles, values, linestyles_radar[idx],
            linewidth=2.5, color=colors_radar[idx], label=cat_name)
    ax.fill(angles, values, alpha=0.1, color=colors_radar[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=9)
ax.set_title('Comprehensive Catalyst Comparison\n(normalized scores, 1 = best)',
             pad=20, fontsize=14)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Decision framework
print("Catalyst Selection Decision Framework")
print("=" * 65)
print()
print("Application 1: Achiral bulk chemical production")
print("  Priority: TON, E-factor, Cost")
print("  -> Choose Pd/SPhos (highest TON, lowest E-factor and cost)")
print()
print("Application 2: Chiral pharmaceutical intermediate")
print("  Priority: ee, TON (cost tolerance higher)")
print("  -> Choose Pd-BINAP (only option with enantioselectivity)")
print()
print("Application 3: Quick screening / teaching lab")
print("  Priority: simplicity, availability")
print("  -> Pd(PPh3)4 is simplest but least efficient")
print()
print("Key insight: No single catalyst is 'best' for all applications.")
print("The optimal choice depends on the specific requirements.")

### Interpretation of the Radar Chart

1. **Pd/SPhos** (orange dashes) dominates for achiral applications: highest TON, TOF, and lowest waste/cost.

2. **Pd-BINAP** (green dash-dot) is essential when enantioselectivity is required, but comes with higher cost and lower TON.

3. **Pd(PPh$_3$)$_4$** (blue solid) scores poorly on every metric except simplicity (not shown). It remains useful for screening but is not suitable for scale-up.

This multi-dimensional comparison connects the concepts from the entire course: TON/TOF from kinetics (Chapter 2-3), selectivity from transition state theory (Chapter 2), and sustainability from reactor design (Chapter 3).

---
## Exercises

### Exercise 1: Sensitivity of TON to Deactivation Rate Uncertainty

The fitted deactivation rate constant $k_d$ has an associated uncertainty. Since $\text{TON} = \text{TOF}_0 / k_d$, a small error in $k_d$ can cause a large error in the predicted TON.

**Task:**
1. Generate an array of $k_d$ values spanning $k_d \pm 3\sigma_{k_d}$ (where $\sigma_{k_d}$ is the fitted uncertainty)
2. Compute TON for each $k_d$ value (holding TOF$_0$ fixed at its fitted value)
3. Plot TON vs $k_d$ and mark the fitted value with its uncertainty band
4. How does a 20% increase in $k_d$ affect the predicted TON?

In [ ]:
# YOUR CODE HERE
# Use TOF0_fit and kd_fit from the deactivation fitting above
# kd_err gives the standard error of kd
#
# 1. Create kd_range = np.linspace(kd_fit - 3*kd_err, kd_fit + 3*kd_err, 100)
# 2. Compute ton_range = TOF0_fit / kd_range
# 3. Plot and annotate

pass  # Replace with your implementation

### Exercise 2: Temperature Threshold for Pharmaceutical ee

For a catalyst with $\Delta\Delta G^{\ddagger} = 8$ kJ/mol:

**Task:**
1. Plot ee vs temperature (from $-100$ $^\circ$C to $100$ $^\circ$C)
2. Find the temperature at which ee drops below 95% and below 99%
3. Mark these thresholds on the plot
4. Repeat for $\Delta\Delta G^{\ddagger} = 4$, $6$, $10$ kJ/mol on the same plot

In [ ]:
# YOUR CODE HERE
# Use the ee_from_ddG function and critical_temperature function defined above
#
# 1. T_range_C = np.linspace(-100, 100, 500)
# 2. T_range_K = T_range_C + 273.15
# 3. For each ddG value, compute ee and plot
# 4. Use critical_temperature() for the threshold values

pass  # Replace with your implementation

### Exercise 3: Atom Economy for Heck vs Suzuki Coupling

Compare two catalytic routes to trans-stilbene (C$_6$H$_5$--CH=CH--C$_6$H$_5$, MW = 180.25 g/mol):

**Route A: Heck Reaction**
$$\text{C}_6\text{H}_5\text{Br} + \text{CH}_2\text{=CH--C}_6\text{H}_5 \xrightarrow{\text{Pd cat, base}} \text{trans-stilbene} + \text{HBr}$$

- MW(PhBr) = 157.01, MW(styrene) = 104.15, MW(HBr) = 80.91

**Route B: Suzuki Coupling with vinyl partner**
$$\text{C}_6\text{H}_5\text{Br} + \text{(E)-PhCH=CH--B(OH)}_2 \xrightarrow{\text{Pd cat, base}} \text{trans-stilbene} + \text{B(OH)}_3 + \text{NaBr}$$

- MW(PhBr) = 157.01, MW(vinyl boronic acid) = 147.99

**Task:**
1. Calculate atom economy for both routes
2. Which route has better atom economy? Why?
3. Discuss: Does atom economy alone capture the full environmental picture?

In [ ]:
# YOUR CODE HERE
# Use the atom_economy function defined above
#
# Route A (Heck): atom_economy(180.25, 157.01 + 104.15)
# Route B (Suzuki): atom_economy(180.25, 157.01 + 147.99)

pass  # Replace with your implementation

---
## Reflection Questions

1. **TON sensitivity**: In the deactivation model, TON = TOF$_0$/k$_d$. If both TOF$_0$ and k$_d$ have 10% uncertainty, how does the uncertainty in TON compare? What does this imply for catalyst development---should researchers prioritize improving activity (TOF$_0$) or stability (reducing k$_d$)?

2. **Temperature tradeoff in asymmetric catalysis**: Lowering temperature improves ee but also slows the reaction. Using the Arrhenius equation from Chapter 2, estimate how much longer a reaction takes at $-78$ $^\circ$C compared to 25 $^\circ$C, assuming $E_a = 60$ kJ/mol. Is the selectivity improvement worth the rate penalty?

3. **Atom economy vs E-factor**: A reaction with 100% atom economy can still have a large E-factor. Give an example of how this can happen, and discuss which metric is more useful for evaluating the sustainability of an industrial process.

4. **Connections across the course**: The radar chart compares catalysts across multiple dimensions. How do each of the five metrics (TON, TOF, ee, E-factor, cost) connect to specific topics covered earlier in the course? Identify the chapter or concept that underlies each metric.

5. **Industrial catalyst design**: A pharmaceutical company needs ee > 99% and TON > 10,000. Using the equations from Parts 2 and 3, what minimum $\Delta\Delta G^{\ddagger}$ is needed (at 25 $^\circ$C), and what maximum $k_d$ is acceptable if TOF$_0$ = 1000 h$^{-1}$?

---
## Summary

In this capstone notebook, we have:

1. **Calculated TON and TOF** from experimental Suzuki coupling data and demonstrated that TOF is an intrinsic catalyst property while TON depends on loading.

2. **Predicted enantiomeric excess** from transition state energy differences, reproducing the lecture notes table and creating a comprehensive temperature-selectivity heatmap.

3. **Modeled catalyst deactivation** using first-order kinetics, extracted parameters from noisy data via curve fitting, and computed total catalyst productivity (TON = TOF$_0$/k$_d$).

4. **Evaluated green chemistry metrics** (atom economy, E-factor) for competing synthetic routes, demonstrating the superiority of catalytic methods.

5. **Built a comprehensive evaluation framework** using radar charts that integrates activity, durability, selectivity, sustainability, and cost.

### Course Integration

This notebook connects to every preceding chapter:
- **Adsorption (Ch 2)**: Competitive binding governs product inhibition and catalyst poisoning
- **Kinetics (Ch 2)**: Langmuir-Hinshelwood framework applies to catalytic cycles
- **Temperature (Ch 2)**: Arrhenius/Eyring analysis underlies both rate and selectivity
- **Reactors (Ch 3)**: Batch reactor design equations govern synthetic catalysis
- **Characterization (Ch 3)**: Catalyst analysis techniques verify catalyst integrity
- **Transport (Ch 4)**: Effectiveness factors apply to immobilized catalysts
- **Selectivity (Ch 5)**: Yield optimization principles guide catalyst design
- **Zeolites (Ch 6)**: Shape selectivity is a form of catalyst control
- **CNTs (Ch 6)**: Enhanced transport in nanostructured supports